# 🧬 DNABERT-2 Embedding Diagnostics and Representation Visualization

This notebook acts as a dedicated diagnostic tool to evaluate, analyze, and visualize the representation space of the pre-trained `zhihan1996/DNABERT-2-117M` model on our TF binding dataset (SP1, SP2, SP4, and Negative).

## Why Diagnostics?
Before training complex classifiers (like an mCNN), we need to ensure that the pre-trained DNABERT-2 backbone is loading correctly with its original pre-trained weights (and not randomly initialized due to Hugging Face weight loading bugs) and that its representations are diverse and biologically meaningful. 

## Diagnostic Methods
1. **CLS Embedding Extraction**: Extracts the contextual representation of the sequences' start token.
2. **Dimensionality Reduction (PCA & t-SNE)**: Projects the 768-dimensional representations into 2D spaces to check if the sequences cluster by transcription factor binding target.
3. **Cosine Similarity Matrix**: Computes the mean cosine similarities between classes to measure distance and identify potential representation collapse.

### 1. Colab / Kaggle Setup
Clones the repository and installs dynamic dependencies.

In [ ]:
# Install dependencies
!pip install -q transformers einops scikit-learn matplotlib seaborn safetensors huggingface_hub

# Detect environment and clone/pull repo
import os, subprocess, sys

REPO_URL = "https://github.com/JustinYuanZe/SP1_TF_Biding_Project.git"
REPO_NAME = "SP1_TF_Biding_Project"

if os.path.basename(os.getcwd()) == REPO_NAME:
    os.chdir("..")

if not os.path.isdir(REPO_NAME):
    print("Cloning GitHub repository...")
    subprocess.run(["git", "clone", REPO_URL], check=True)
else:
    print(f"Repository '{REPO_NAME}' already exists, pulling latest...")
    subprocess.run(["git", "-C", REPO_NAME, "pull"], check=True)

os.chdir(REPO_NAME)
print(f"Current Working Directory: {os.getcwd()}")

# Ensure src/ is on path
if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())

### 2. Loading FASTA Data
Load 101 bp sequences from `data/processed/`.

In [ ]:
sp1_path = os.path.join("data", "processed", "sp1_positive_final.fasta")
sp2_path = os.path.join("data", "processed", "sp2_positive_final.fasta")
sp4_path = os.path.join("data", "processed", "sp4_positive_final.fasta")
neg_path = os.path.join("data", "processed", "negative_final.fasta")

def load_fasta(path):
    seqs = []
    with open(path, 'r') as f:
        for line in f:
            line = line.strip()
            if not line.startswith('>'):
                seqs.append(line.upper())
    return seqs

print("Loading sequences...")
seqs_dict = {
    'SP1': load_fasta(sp1_path),
    'SP2': load_fasta(sp2_path),
    'SP4': load_fasta(sp4_path),
    'Negative': load_fasta(neg_path)
}

for name, seqs in seqs_dict.items():
    print(f"  {name}: {len(seqs)} sequences (length={len(seqs[0])})")

### 3. Running Diagnostics & Embedding Visualizations
We now load the DNABERT-2 model and run the diagnostic pipeline. This extracts embeddings, performs dimensionality reduction, computes cosine similarities, and checks representation health.

In [ ]:
from src.dnabert_diagnostics import run_dnabert_diagnostics
from IPython.display import Image, display

# Run diagnostics (automatically falls back to CPU if GPU has compatibility errors)
device = "cuda" if torch.cuda.is_available() else "cpu"
sim_matrix = run_dnabert_diagnostics(seqs_dict, device=device, save_dir='figures')

# Display the resulting PCA / t-SNE plot inline
print("\n--- Visualizing Diagnostic Projections ---")
display(Image('figures/dnabert_embedding_diagnostics.png'))